# Gemma 4 Developer Agent — Phase 0 checks, EXP-000 gold control, baseline eval (4x L4)

One notebook, three switches (`RUN_PHASE0_CHECKS`, `RUN_GOLD_CONTROL`, `RUN_AGENT`). It mirrors the
organizers' Getting-Started recipe (wheelhouse install, `VllmServer`, `swegemma.evaluate.Evaluator` with the
subprocess sandbox) and writes everything under `/kaggle/working/runs/<EXP_ID>/` in the layout that
`tools/analyze_traces.py` and `tools/estimate_budget.py` read.

**Attach as inputs** (Add Input): competition `gemma-4-developer-agent`, dataset
`metric/gemma-4-developer-agent-wheelhouse`, model `google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2`
(or the `.../gemma-4-e4b-it-qat-w4a16-ct/2` / `.../gemma-4-e2b-it-qat-w4a16-ct/2` input when the zip's agent.yaml names a
small variant; the vLLM cell takes the model from the zip).
**Accelerator**: NVIDIA L4 x4 (competition-attached notebooks only). **Internet**: off.
The submission zip comes either from a private Kaggle dataset you upload (`SUBMISSION_ZIP`) or embedded as
base64 by `python tools/build_notebook.py --embed dist/submission.zip`.

**Status: UNTESTED on Kaggle.** This file was written without access to kaggle.com, the wheelhouse or the
harness source. Cells marked `# VERIFY` use API names taken from a community mirror of the starter notebook
and may need adjusting against the real `getting-started-gemma-4-developer-agent` notebook. Read the
starter notebook first and diff the server/evaluator cells.

Edit the parameters in the next cell, save a notebook *version* per experiment, then download
`/kaggle/working/runs/<EXP_ID>.zip` and run `python tools/analyze_traces.py runs/<EXP_ID>`.

In [ ]:
# ---------------------------------------------------------------- parameters (edit me)
EXP_ID = "EXP-052-t2"   # overridden by --set
RUN_PHASE0_CHECKS = True          # Q1/Q2/Q3/Q7/Q9 answers + HARNESS_README copy + sample_submission dump
RUN_GOLD_CONTROL = False   # overridden by --set
RUN_AGENT = True                  # run our submission on the sound tasks
MEASURE_DECODE_SPEED = True       # Q10: one timed completion against the server

# Task selection: explicit ids win; otherwise the first N_TASKS of the (optionally repo-filtered) dataset.
TASK_IDS = ["fastapi_14301", "fastapi_14448", "fastapi_14605", "httpx_3672", "requests_7205", "requests_7309", "rich_3006", "rich_3468"]   # overridden by --set
N_TASKS = 8   # overridden by --set
REPO_FILTER = ""                  # e.g. "Textualize/rich"; "" = all repos
EXCLUDE_IDS = []                  # known-unsound tasks from a previous gold control (experiments/README.md Q5)

# Where the submission comes from (first that exists wins). Upload dist/submission.zip as a private dataset,
# or embed it: python tools/build_notebook.py --embed dist/submission.zip
SUBMISSION_ZIP = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-sub-v1/submission.zip"   # VERIFY path
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVyPXcNCvgEAADEDAAAKAAAAYWdlbnQueWFtbIVSPW/bMBTc9SsOyOIApis7TgIoU4tm6xA0GQoUgfEsPUtEKVJ5pKPq35ekXCfI0C4CRd473gcvcP/jQZXXG2zKzY1ar1V5W0GcC6g7si17HMT1cJbxzfSfW7YBi9DxPHd1hfvtF3htW8Mwzg2XCA6P/HKMQE0mD/wUpoZliQcSMoZPu9dohA6BxT8vQcNgNMszFl9VWd4iCDOeNssZgwNZ5Y4BFJAu50YHDE7bcAff0cB4hDvgrCv6GYSVcKt9EAra2ctVcYGnCJjNkYiO5qxD7xo2yyjbGQ8n0DaOHOs0cwd+ZZnenGsfj+GP+x2lf/8JZJszmQ6RYLQzIxYt9z2preLtXumgXiiocUvrG1WH6NefE0wUeb0to0hLPVfwI89XFPm7qw15X30MtmjY16KHpLXC95hyjkB4cF4HF5WPOnQgzAXEw5hf50zj885fz9ZM2E/vjS9hOIauXzljlGHvMYpOZYGp7k61pGfBv6kOcyP5qaQUkgKJLqMam6udsrA8lCN0kvRkdYnj1P4ssO64/nW6JmUT4+4j6ap4y70qAIXa2YNudwOFrnpfyux2NVFv/onLcv4PO4mbgX8AUEsDBBQAAAAIAAAAIVzY/KEY1QAAABgBAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1sPY4xT8QwDIX3/ApLLCC1Vdq7KzQbAxtIjDBVucZporZOlDoC/j2hJzHZ7/Pz07uDl4/3Wl466GTX121by0cF7PDGTyfY9RZXTzNYv2IFmSanaUZTwe50QgPXn8OfUBtMoMkcUsfyVfR9oPVmMElbxrSD8dZiqsBTycUy/s4+Qfgi+Hx+e31oBOMWMWnOCRW0jRQc4hgVyGa4HPuioD+LTX+PIXPMPHJYkHYFT+3QCXaeltJ5nAJZPysB8I+u2czICs5y6Av3NK3Z4Mgu5NlxSeCUUfwCUEsDBBQAAAAIAAAAIVzoDefWxAEAABkDAAAQAAAAZXZhbF9jb25maWcueWFtbF1SS4/aQAy+8yss7WEXiaAkbEoIUi+rPbeHHnpDJjEwamYmGk9g99/XnhD6uCXf2P4e9hO8//ye5VUJZV5+yYoiy7cNWPw4RO/7Q4t9z1DWkH2F1wrQddPbGBxDnStclPkeorHkx3hgar3rpOM1/1Msbwdr3BhJegADTZybzTOvF0/wphwNxAtBIOwoPDPcfOAIlpDHQB0MyAyGodhA60cXBZqUvaRJQibE0XgHu3W9TMw6DoehNxTAEYkmPIpC2MAgSBfwFOGFOhNX0F6o/bWC0XV+CeyhqODkA5zMlaZChqEfhR2il37rA+01jZ7wKpaC9zY1HH283JW3yPKCEbxr6a5yUy7V7g/NrgG6UviUWR31+u2k9HQH8ay/yag45BudyVoUQIx/xPXw2ZTbbVbudss1fHP0SGcyWKtKsTBlCWlVK6gmJzOQIqq0sswfMc2llQpPPva6XfEpOlAGcsQQ53WE0Wm/kTeSYcZxFEZ1cSTjztCKFmu67D59pcnqkEwb+eJvotmlNcWAj5BklQENC4lcV5KXp2y1jtGmE2HvJEm6Yj+iLr1ZwP8H2OgFCvzvITeytRn86ygbqGd02o2YXvwGUEsDBBQAAAAIAAAAIVxq31bzJAcAAG8PAAASAAAAcHJvbXB0cy9hcHBsaWVyLm1kdVfBkts2Er3rK7qSQ7IpShPPjA/xJTXx2NnUppyU46pUTkOIbEookQAXAK3Rbu2/7+sGSMpe72WGAkB09+t+r5t/+YlMYDKO+HnkkMhMyTs/+ClS9F06yy67g3XMYUfvOfr+I1M6Mk2RA9kYJ6azTUddi4Ppe44J706hYWqOxh3kuMGSSTZ2lmM+mUzilvZ8NB+tD7vN5gNWj7Zt2dEhmBaXm3Hs5fwFl9HBJmpt11HyZKgLHI8UePTRJh8uiKClMLlINkXyZ0cJbsQd/ZLI8UdcFnm+KbCJ3ll32NED4h7GdMk3x8bjWvoXB48twq/AzRLL9RGJIJl4IlztyHb54t43plfDdDYRdlryQbaKCwEwG91HuF9/TT9N7YGTBq6XDRyjORRsirPJ+36La3uNMNmBaa+viYeIRHHCTmeRlnOwiUNUQ3vu4KpcUuVXYSWf/yaSXIi/fnIJFpDkvB/N1fV/HhEaI7Ow0NsB8B+BbaUHzz6cACClwIwayAlryQj8+J2BbfE8YJ2sy46rvXiNGbWenE/iWHOkOO1h5mk0qTnu6K11Fjk2UojeyX9TLoVBXBk99cApKHY4lCxAQjWxGssIf2AzaJVIZQOENpguxc1PCzaCdUAhAqMZTT+y44xH41vOlaVXSD6dbmjZV/8LvIahRvQsqPNsUEHqdRf8kJmQb9sREm9hb0rjhG3f9/4c4ff77Ec+9Wrz77E37sf/bDaPci+9wIpaeHqxLt4ui7fr4t2yeLcu3i+L9+viy2XxpSw+gNtm33OOBJgCenr7y69vUAdOw6bffn2kPer9pBgYevfmz/wbjDs4AdeUtwv3L5He/UaP7x/efqgkgUq7ChlsTvH6/UooI3jFQhUA3JdsvtjR66P3MetPc2Q5XurcuItyRUt788iNReokA4Ax+HZqkkXucHTNX8ksdEyqVraEpZfs9yuYj0epm3q8pCPe3Tb01W63+6rOIfEzh8bGIgZZ6NpcMYscWieZXQ22HJtg9xw1SvFOLpcgVa1u5O+TxjteaHvS1ScnpNz+s5hdLyuhlOoevRVuJSjXuyw2EzRA1LuYqKvF2K5W+0tgA03OJt1qLRQOr9cZ8VsUKRARO5k6QjwfUJ6bD+Y094G1VnRfFvUMvajotqK7iu4reglGC8TCEET+aiPZFF2rJdsadb0ip7dBqGRZqF/LA4ThCNelNq7qD5u+b59iClCkWpMqJ5aK0hOOz/OJCkkaLXKl3Owvsi/8TaJKbrWOSu5ybsXLDHOEWkHTksQQqDO2RyoPHioSjD0cRUz1FcfPab7lcZa4VJBUlc3k0Hhz7cQsUPPhraiRHs/1f7u4o2KSWyNkS9KsjULpAGQbHL5bfddVFPI51+ncb69ZIOqJ6q/oxDxiQ9ipNuBPTB5L4SJinzO8o5+9hBk5U+p+t7n/zJzikmOTjhcxKOhRv4id2ftCjMUhEePCogr12PorT/SqWiYANSDvbreqSMKK4KcDtoHEU+OHAW7nKjj4L6Rj8/KLznqHQthzY0QNfEcDsJGYO/ucpiCEtYPyjEPwIfN3RJb42aLx4OAe+TqheUt5grfSXrXxe0Qhk8Y/PsG2+gKw1ZXTV+AKDe9KN0bx5boZBdR2s5VQREbgzSAqWEhYqgOtHNOP7OfzlW5cSlUF0F32VFbmN8zBgAMZ75WXu2xpMb9Un1KjllItBJb6m5tk4INdHrWliZS1OkE4sOiUBfqa/75MUjrAiV+tBzHOs/At88vrvz+8+zm3I+0t4uDvgTvYuHb7U2UYppgKeWdezRqArLmmn1rJhnUYQTGBwXfpN2WAvZ5tR1O8Q030PLAKr44LDjrQXHMevIJFyQ+esYWHyQXudfiVZiGeZ8FuEF3iuf1RreL/HTqBKPd3Tzo26g+0Oi0qYYkEnHvHqu87DE5F4eFRN7+4eGUdhh7kbUTvRcUqeA9on+hNCo70qDGhElnlvL5Jw/h/eDY3G51/lgsEWXkVhVzfyKgIxBpJx1vd6LlLs9JeTfBgnx+kVMEynxmqI7cOWSCpDpnCJaUENOcPHRfh/F86fBukxmTaaKb3MoNogYof17MlwNEZ9Cx1uCrBZzN7PNlxZGkmC3I5TNzae1WHOAEGuO5Dnj9EZgRdkW6/No88qgvzhFiiXmadVWQYSpp6nZ8n1+ncK7/K1J5kiMUpkXrlZ6HjgpDUHZaXiUn6ZO4Mn2tmnblp8hm1kCn8CTyfWKnxMfAkXyRTrEsjQRKl4MGCH74XqfKuFeQGvARUqi9eaQdQ3yLK/pITqIUDE6+9Mo/X8UZnKMRreuH7Zf4O29EbHc/0Uwmfo63EW77NBg8gbKOkLRMpdG+bv+3y9wRM/cEmSOeeUpTp8Lo26dv6ZopBSHSDsuA+6qMfU/03pJ/xTQBVaORrVL+Xi2uFStzq9XKofFrBjX3wRoaMsfchq8nmv1BLAwQUAAAACAAAACFcw1YkemsCAACABAAAFAAAAHByb21wdHMvZHJhZnRlcl8xLm1kXVNNT9wwEL37V8ytlyUSrXpZISrKLrASWiRUCXE0yezGwrFT2yFEVf9739gJi3rJx8zY8+a9N89+IB2YxmASBzonf6DvZFzDPePh0pyJFT37IZB2R8trSi1T7LS1HBP1PkbzYpnqFmmu6Dq/SUc68EjWOI7yMxprqfEr0ilfwO+6xmmra87/gXXD4Uuk67ur/e02H8TlxqVISY65Bu+hbsn51Bp3JLaRK6WAjFr9xoijwNuYS7sB2FBJNXAiMlX0k60fySB/aha49yGRdwhGf0ij0GFiHOYx8+cqfx6M5RnGB/x6CEFoSvyehLwyyNHgvuQXRpT6gzHdj79KPQmd6PZxHteZRiPGjRFeQI4Glr63BoJkznodkQcPXlKlDmPS4Mr9DRo8DtBirc7o4X4jE9a+N9wIgIA2uOngw+e/4Dt63N7uHvYrNEktGbBssuQ6AX1Fu1SYqr1L5jj4IdKL9fWrTPlNhjv/WrQtPBWZJ0hRQEFYnJBB0TiN/oR2Lo0ePcnXYFDq7IRHDffs+Q0Ajavt0MBExQaZF8ARPY2bkVcYd799krhMncfIFwkex9wUAtAzruRUdq3u+POcFNNkxbO+n0g6T4gEMdeH+NDz92ACIGfJ7CR9Nz5jKWJgC7Agc0g36Oq7jsW3+sUPxe2LFc5ot7gkG6/VMtU8kXAV9RTpZne/XYOyV+dHtypLWPAb2cBCyv6BNo9XN78WSzra7jfV4rGp7GscM5sLdtSBLpihy6f+26TSVvr0GmQC2mkx6SgrZpIC12t1cXGhJOftrKe6vLxUUOOUcsvy59TT3fM62yGCGIbSCmDVP1BLAwQUAAAACAAAACFcx6EOtKQCAADwBAAAFAAAAHByb21wdHMvZHJhZnRlcl8yLm1kTVTLbtswELzrK/bWi2OgKXoxghRt7DyAwAGCAkGOtLSyiNBclUtGEYr+e4eUHPtiiK+d2ZlZv0oiE5iGYCMHuiRp6TtZ33DP+PFxPtElvUoKZPze8Ypix1SbpLykW1zGg7IVeG/F09AxSuaNIYjf07txiUnCvNxxZ94tllaptUEj9UGaVHOzQP2G6g4oTDbmEoEXxO/sybYzhGk4fFG6uf+5vduQs56pYa2D3bGSQcUPalPIT6mRwWvEk8OSHtpzitrJoOSF2ARncVXRXQ2sGTxf7Z2p+QyUvDmwLqsKShBa4Pw+ijgttA8JnXiJEMY57IxL+sVOhtymOfEO3EuIBA4GoG0csvxWNc2yls9F+Wyt40mSvOIPU6N4CiHbEvkjZrPOeooyswfFvyDvf/yrqpdsH9A+36OcbQz2uCkKmwgEMn1fZBgsuPdGcW49CuJouoc2KfmpfgOA5+RYV9UFPT2uc4e19JaLdwEwqNTC4LNVkAM9b+4enrYLgMQO9mrJmY8mgj0MipNStfho90mS0s5J/Za7/Jab+3pZ3NZJp/JJI6yYLesdXuRGARwHObGdr6rkSEkNBfM9N+KnRoC3iBey6GuXGgZ+SVTRxWrxE+GemC/R7nbzkvdz16WNUijz8cwNH8Ori+NIKDJz3idpHB1Qb6Qfc7DDiJ1gMRaf5sPPP8kGUC6WuTHjrqVwmcxgjRjIecs0QJXDAQiQbycpTuM5R+HilPwSvM6U3E8dZa3UjEq3D4+bFSR78xiZxTT0E3+bJ34SZftE6+eft7+PkfS02a6Xx4yN0/+DDkXNI3fcy2Mu4VBegXCXm2WnOaYTbBk2AzFB7Wzc9nnEbKyg9aq6urqq8pm42c/q+vq6ghunI8/D2dHL/euqxEEhDMPpCmSr/1BLAwQUAAAACAAAACFcHDFqHIICAACkBAAAFAAAAHByb21wdHMvZHJhZnRlcl8zLm1kTVTBbtswDL37K3jbJTWwFbsERYdtSbcARQoUA4oeVZuxhSqiJ8l1jWH/vkfJWXJIoFAi+d7jY55lJBOYpmATB7omOdBnsr7lgfHl03ITa3qWMZDxneM1pZ6pG01oa9rpK3uYc0w8I3kYE0mgmEziHLYxjkzRzJE64Yia4rsVirVkWnzoJRjf9JrU9Ny80gEnfjdNclrXJGpM5BU5Nm/Wd6U9ew7G0WBSTyOyAY3bkvnGYSbBq1DQ1FUF9NSbNyYvlERczN2PY0yIaH3nEJlr+sZOJiAGqsCm5fAh4jBIACePYJRDmlSyTGp95rfKx4N1XJjpr8yBmjEElTLxe1KB9SZwZ1EvCRXkgPhncMZ/+VtVTyp51nLJRznbqpjc2lQEMcAyDM6C4mSdyhCTao+CuCrvQPOsDBo8jo7jurqih/uNMmxksJAMDwLaoNIhD+D/ryBHetz+2D3sV2gCmW2K2RsekwV6DD8VpRqBB7pRxkgvTjBAsLxWch8/kbMeXTPpfKQZoyigaHDIUKJonCa5mGN5GgU9SRooqO/gBvEN17TXEQNK48aW0V9fF10AR+dp/YK8Bt399knjyjrTyIUUj2duiwDoGVeapeFojnzJE06eHbp+l2FezBVTOBmxmDvw79EGQF5sq303krGUYXBMWKIlpK5v5HhEB8j3ImPKpU5WuKLdySXZeL1RVgujvFq6Sne7++0akr16mfyqLGrBb3VLiyj7B9o8fr37dbKkp+1+U588NpedjlNW82LlIBfMcMxZANwrWXZRbVraap+8eoBWkOquUKcrZlMFrdfVzc1Nlf8V3DLP6vb2tsI0zleep4urp5/P62yHCGEYk64AtvoHUEsDBBQAAAAIAAAAIVyZg86QoAIAAOYEAAAUAAAAcHJvbXB0cy9kcmFmdGVyXzQubWRlVE1rGzEQve+vmFsvzkK/LiaktLHTGoIDIRBylHfHuyKyZitps1lK/3vfSHYc6MVoZyS9j3nyk4xkAtMUbOJAX0j29JWsb3lg/Ph07MSanmQMZHzneEmpZ2rEp2CatKBJQkt7CXlR04N5ZuIXDjPF+bATt6BoO2/SGHhBB47RdFjEhErENS3j2pYCY4OnF+NGBoBJGcXGiM/Av0cbOC7yzoMCFAo4OwRpxwaIryDjZjQkotUbJZeZX/fcPBObpj9d1AI9WN+R6Yz1MdGs4rbrR3LWc6QdQw5rFYBx4lBXFfRTb16YvFAScbFwGXHaS6LGOIfKXNMPdjKBOBnAmZbDh4jFICGReBSj7NOkpmdty7PMRV7ureOiU7+yKmrGEHQYiV+Tjkg7gTuL+5KoVt8xKP4ZnPHf/lbVow4NaG/ncZ1tDWrc2lTcNeAyDM5i7JMF98FE9K3HhWiVfZBJoy/3twC4Hx3HZXVBd7crVdjIYGHmm9k5Be++ghzofv1zc7dFSmzqyaaY0+UxfLCvaZOKU5om242CROycYFxQ+VnFffxUZlJ8KuPRuRRSNDicUKEATpOc2R63RgEmSQMHdR/yIb7hmrYaUFBp3Kj5y7uLL6Cj87T+yLyGXE0G6qo6y8gXKR/P3BYDgIl44pSWoznwe52I2+yAei3DfHoaJYD/Z/wUZMVdSeZShsExIczHkmmBKocDEGDfTsbyXE5RuKDNKSU5eL1RVUdF6lU0c6Sbze16CcuevUx+UZ564W/1nRdTtne0uv9+83CKpKf1dlWfMjaXfwV9Iyr+/AhhF8JwyKdAuFex7KLGtMAqzmBgJqgVpvpWqNMnZlMFr5fV5eVlpT1xx3lWV1dXFaZxbnme3rUefz0tcxwijGFMugLZ6h9QSwMEFAAAAAgAAAAhXABjneynAgAAAwUAABQAAABwcm9tcHRzL2RyYWZ0ZXJfNS5tZG1UTWvcMBC961fMrZeNoS29LCGl6W7ahXQDoRBy1NqzaxFZciU5jin9730j2dkUejG2Rpr3MU9+9APpwDQGkzjQJ/JHPIxruGc8XJorsaJHPwTS7mR5TZoaczxykA2x1T1XdCNln1p0mY/QaKylTj8xYZn84dn4IVLdogkO/JCCnj+xQyc6cYp5b9Qd04FbjSOBDtM/gMEPSTi4ghaTTtxlJikMdRoCr16Lx8HVyXgHSblx4BO+VoSumVPf+wiydAja1S2WGw4V7RJ1Q5SGosB0vS0Ao5CUcybGQZr9GkzgWCkFcwhsmZyn5L2NYNCUJmBCtUYf7aaKrtn6EeehKLAG2ruIl96HRGCpKfpjGmUiGWJ9Rlvl16OxWV2Tv/hF12g+hOxL4pck8zvrBJXFbqV+91a7z3+UepDxAO31PNqZBiYSNyaVSWhw6XtrZJriQa8j6sahIUplH2QS7M39GwDcD5bjWl3Q3e1GFNa+N9wIgQAYGQZcf/MVfEf322+7u/0KIKklg+mbnDotM8tzyE7V3iVzGiQ9B+vrJ1H5UcS9/0DWOKBm0fmVJoxiDlVvcUKEyrhHf2Y7b40emORrOCj77IRHjWju+RkEjavt0EhGZXfxBXRknkhTYV5B7n77IOuiOsvIjYSPY26KAcCMqyWDOdxvdCJmkwXqV99PJMiTJNm403+iVkZmJ8Hd+MylDINjwh2dl3QDVN9JZmHfAfclt1qicEG7JSU5eK0WVbMi8SrqKdLN7na7hmVPzo+4MflSF/5GfgLFlP0dbe6/3PxcIulou99US8am8suIY3Zz4Y59sAth6PIpuakilm2UmBZYwek1zNRxZip3hU5yxUxS8HqtLi8vVb7Edp6nurq6UpjGueR4fFN6+P64znGIMIYxaQWy6i9QSwMEFAAAAAgAAAAhXH2OOo76BAAAPAoAABEAAABwcm9tcHRzL3JlYWRlci5tZHVWwW7cNhC971cMkkPaQqs6LtCDb4bjJEYDO4gdFDlZXImrJUyRCknteoF+fN8MKa3ToBdb4g45M2/ee9Q3P5EKmtJO09aEmCgm1WvyW1KUtBpq+oaQoFUnMUGPPprkw5GU6+gQTNKI5OWQarrVz6nCQXudfwqRtGp31AW1TeSdJv2s2kS6M4m2wQ909FMo2ys5ElkcHkiNozU6UApGR8kth8Sa7pw98j5qEeZHhG+N1bGi6OmwUyn/5scjGZf8UnZIZCIpa2XFKlSXm410MFjVe17Q0jtHtL7T9WrF7Q/qSM4nanfKARzljmlnXF/T16ipYXAeuYSGkO5U0Pzz5B5bPwxorgEEqH3rgyC6lrcx+D6oIV5Q0wOVHrXS2jUVNTby36i78r7DHv5/aBuGmuvNA8B5g9/L/05bLSPhEipyEhWAd9DA3U9pnFLGZQ5h0EvY5DDzmCIfFNtgRsC9Wr1+TW9ruvIuBcyuoo1GA4IC+vUWY7B29WVmiIlx0nLok0YniuKOobcG3CrIZg7E47DxlsdmeqfSFHiEOgTOjaG7Hq8fHx4+85DSFGUeMZcbNMId7ZWdMD2AFvT3CR1GLhNzR5KBh8X1oi/Ge3mr6QElAPFO86pun5hdHpMqWXN59pg7P6+JW4uikpKmY1KEoheMC0McJjxuGPcFkkook6CImi5nBvK8EhOEuf8CLgttVOAhKsF81A+k4sNAnKJRq4XfTg3YAEIfgnf9BUCIyNEbxEEWnHlTgK6E2ccsH3BJtLeZjGVBMnjC/q0yluk7qNTuFiVCTx4J2QgkAgC/89LXaFXpYtAD7KBereleqwCxe9dqaeUHQtOr+8/XVzfvb64eH+7+ur59Res1vfmtHo9v6B9iatP6/KzJouHUg2c3GnVrtqYl02mX8IShYZovafIfLLmQO9agnMHdcEyBJld1whaNppzo7fkZOOpAp5F5wZD/0oB5IT3yciOINNp1+fXX7Isydj9ny/t5OgLp98knKWe2kM6LBW69tf6wnka4TQYs2wGZ7Qsf3pmU3SLotX5m8mQXzLLhPdnYlBVlIPmetQDjlOq1ZL4s3UXz/BOlJDj5pOwy1GzzcIDZwMr6XOeUIuZAze8HH57iqFrdZJX8UdPf4kQvzFZFGDIwVxmSndrPP/MoVvfJj5KQx3Ng01+iGFDdLSaMB54SQ50NWCTsqOdr5piV6ACDcvGA2aGromBshECg/iGbnBfXJpZBKJdATDI1dHFzf//1+kLuMjYcPYBvs2FlkTK2M1vOzggYdPAjCJEPncUZJVXxNmzZe7tHL7zIV9IcOjvJz6bz/uYTyuCYUSUoMWhcVdxpucoW6Ku5OuZUBAxtsQcEFpxWn25ur+8vXrCK61i6JjcNG76jy0F5MhVPzGRGv+QMcJlsqldfrj/c3N1erE5O3k4hMFrsdfNZ28m1iSUHom6sb59OZVWMhEHv/+PQJ0H/nP4k2D+LXmfmGWEcbIlL50JE66IP6XWDx6dY5RWRZlxczgfTC4WMY5tRSWBg5hccdGzViIqFS0JHXhWrz3XxLXfAZXn18fL2A+bHM2GJHjzE4+D4LdJFdcxkL/vFiYXXcAb5nNrpoH+k1AzjXOomKNfu6tX17Ttwdlu+dSYr9D5ZEYyr12mZaJd13OO6XABrMtMm9+T8wWWDEyqVevl2WD6nsgPorsrfZ8her/4FUEsDBBQAAAAIAAAAIVx5tw9abwIAAHMEAAAXAAAAc3ViX2FnZW50cy9hcHBsaWVyLnlhbWxtVMGO0zAQvecrBu2BrdRk29AtbDggJDggJMRhD9wiN540Vh07eCaU1Yp/Z2yny4K4tJ7J+M28Ny+5go/fvpab2xrqTb0vt9ty87oBHjDnX70C6/0E6oiO4ZrUiMDeW1pDOsvPZI07Qm8sruBseAAFU/DjxIKiGDoVgkFKkAGVxvCS5DD5wKCcTvne/EDQQfUc64KfjwM8Tla5d79SzWN61n5ZQkZrCYw08MDhIUKMYBz4IOhVcSXnzs4a2847lrkJNPZqtkLAeQbnHa4yRzXJ9BhgnImBEFOSFZ1gRCIhDdcXjGp6aG73d+Xtm42M6qc0wOykYwas4BODyTytinAc78d5B281Ac2H0XA7Ke4GUc+LTjkF6PSiz+zgOkndytEJGWl6V2/Ku7peJSiZf4hyByylIt/iIIOLPjKJ4bcCy/gzgcbCs5+tKKZOmZubtQwlMwz/abSvy+2m3q3WcB5MN6QLyyp7Hw5GUxT3XrLvP3yWLrJ6FbtSp5ZRjCMOc8fGyzZcJ/I9y1A7s7FJyK10SXy0F2vEpQihCJNQRBZiwzOjzq5IjJJmh6A6uSGoRgvp5bEK0Yk9VoUTUzaXtRYaqQtmis0buH+yYbok2KgN05Nxsnljro1mXkM3YHciQCVSRDJrOCFOtDg20GLw3IwSnUkRobwb8ZyXm8vT0qti9BptA0ccR1XuStwdSsPld8Xleae2+7Lj4pleDbxYjAxVdZMXQTcLt2rUxb82by4+L9Ir2hQAZfSUFEhDp3Ms72AimKInuik6B8P4Jzwit+JinimFzw1ciHcwyPYvzeN/b45/z5xzdHP5SlQParTFb1BLAwQUAAAACAAAACFcoxFEsaMBAABOAwAAFQAAAHN1Yl9hZ2VudHMvZHJhZnQueWFtbI2SQW/TQBCF7/kVT+qBRLKNkyatcIUQh3JBQj1yswZ7HK+02bV21g0I8d+ZXYdSCQ65rOzZ55n3vvENHr8+lfVhh129uyu327K+bzCYZ0YfaIgcBMZhokDWssX6M97jUOBs4qj1ODL2iB538EN+641MFLtxU+GRuhEjaYMo8GeHIdDJuCMoZukQvIvpu3RvnMQwd9F4V0A8nMcUeDDWwghkpMB9AXL9S7fIp4kDxTkw1nV1W6CuDum4T8e7AtuqfkA3emFt6bwOnR33m2p1s1g7BxNZezkGf6cugnsTUxhhEfUBiRQZs+s5LDTaLw9LyD9ohDkXApOK3og+TD5ErFPx52TJffiVjVrttMn21ciYILAV9W1cZ+ee205RsNNkTu1oTslt8272NZ59mqbOHZ24Wcav6KgftJ0lkQZPlwV9TMVVz9IFMyWYDT6lZRoNMbEeLiX0trSacSEQpABnHrrTF7rkjpaz4VecL8I8/x9wSXtB+h+M1Urmb232LM0KKKGRB3Ns9W8ZG/y9fHuB226rH3SyV0l310tvr5fur5ceFulvUEsDBBQAAAAIAAAAIVwJvygZxgEAACUDAAAZAAAAc3ViX2FnZW50cy9kcmFmdGVyXzEueWFtbIVSwY7TMBC95yse2sOC1GSTbNsVviAkel1xWySEIjeZJlYdO9gTWoT4d2xvWlbLgVNmJjNv3vi9G+y+fM7LTY26rLd5VeXlg0Dn5IHJoYI9YLOCNL0m+FFqTZ7RDqFAKzCNEznJsyOUxX0BtlZ7fP0WBjoo0+q5o6a1hsmwh7GGBBTDE3nwQHAkO3K3PgSTdRxqzs79gF+TlubD7wRjLA/K9CDtqchuMMpzY2eeZm7YHsl43JcPdWqNjcfQ2+znridGVdZrvOUhYob9dE6EOhycHdP+dOc7AfkcQfkQarknjUnPHlZ3zxzoFI498yplEu3My4S2PtyiAvrj7gl7bdvjCt4m9FZOETEckNIfNjypv0W92Rb4uMyfFA+hA20AilfuHj9BKxOG7RzfTPq4rMiMHOmqS1NlHfnWqYmVNQJPTr1U6786ZaPtSAv0NI4yX+e03ueK8++S89NaVtu85UwZz25unxe8WbREUdxN4fEm9ndXLsXYZa+1FknsbNHpSD8X7oF58ogIJsl6MpHVdSp+D6oXGV4yFpFyLNmpmWLyfrNkR4HtOsT/OEIkS8SuiyH+IuO1S0SySfpzueLiGIGDDK7L/gBQSwMEFAAAAAgAAAAhXObOJcEdAQAA4QEAABkAAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMi55YW1sdY/LTsMwEEX3/opBbOs0CWmres8eiQVICEVuPEmtxA/ssQp/T1zaKBtW9rzuPfcRnt9feLmroS7rPa8qXh4EqCB7wgA1uB52G5B2mBA6mSJugNB4DJJSQCiLXQGv0iCQc1PcwEVb5S7zgZrXfQQZF7GqYHbeXNTbmimMXdCetLMC3oJee/7jxoxTOAkY0BjJG47NiWviX5L4pZHVnnfEtI0UUvcn+6BtNyWFUBRbH5zxFLcLQWEUu83bzllCS1GAdRaZS+QTtSP+3Ihn3mtIAR+fbECbqZar/PZ6EAzWxCIj55bzrc/F8V6NAvbN/Dfyu705kRvRzupP5aHOW2dtR22HlfKqeUpqQBJQlXVzndxT0Nml4ZxT9HKKyH4BUEsDBBQAAAAIAAAAIVwXAE/THQEAAOEBAAAZAAAAc3ViX2FnZW50cy9kcmFmdGVyXzMueWFtbHWPy07DMBBF9/6KQWzrNEnTVnjPHokFSAhFJp64VuIH9liFvycpIcqGlT2ve8+9h8fXJ14ea6jL+sSripdnASrKnjDCAXwPxx1Ip0cEnWVUOyC0AaOkHBHK4lzAs7QI5P2YdnA1TvnrdKCgkyGBTKtYVTA3ba7q7YEpTF00gYx3Al6i2Xr+48asVzgK0Git5A3H5oMb4p+S+LWR1Yl3xIxLFHP3K3tnXDdmhVAU+xC9DZT2K0FhFVvmbecdoaMkwHmHzGcKmdoBvxfiifcWUsDbO9PoZqr1an57owWDLbGYkeeWD22Yi4fjUg0CTs30t/KrXZzID+gm9UN5rueti3GDcXqjvGl+ZKWRBFRl3dwmfyno4rO+zCl6OSZkP1BLAwQUAAAACAAAACFcwTtzQiUBAAADAgAAGQAAAHN1Yl9hZ2VudHMvZHJhZnRlcl80LnlhbWyNj0tPhDAUhff9Fde4nTKAzEzs3r2JC02MIR166TTQh+0l6L8XEAlLV72vnvOde3h6e+b5qYQyL8+8KHh+EaCibAkjVOBbOB1AOt0jNN5RlA3B6KOC1selOAChDRglDREhzx4zeJEWgbzv0wFG45QfJwUFjQwJZNrUi4y56XKzqyumMDXRBDLeCXiNZg/xX3tmvcJegEZrJa84VlduiH9K4mMlizNviBmXKA7Nr8+dcU0/KIQsO4bobaB03JAyq9i6r2cCdJQEOO+Q+YHCQHWH32uEKcCSWsD7B9PoZqrt1/y2RgsGe2IxI88jH+qwNKe16wScq6m28qtench36Cb1h/xSzlc34zrj9E55N7wOSiMJKPKyWjZ/KejmB32bU7SyT8h+AFBLAwQUAAAACAAAACFcfKwpticBAAD5AQAAGQAAAHN1Yl9hZ2VudHMvZHJhZnRlcl81LnlhbWyFj8tOwzAQRff+ikFs6zQJSat6zx6JBUgIRW48cazED+yJCn9PUkJUVqw8nsc9997D4+sTz+sSyrw88KLg+VGAirIjjFCD76DegXR6RJCgTNdhREeQehlwB4Q2YJQ0RYQiyzN4lhaBvB/TDi7GKX+ZjxW0MiSQaRMuMubmzY3U1ExhaqMJZLwT8BLNH/7/ZGa9wlGARmslrzhWZ26If0jil0oWB94SMy5RnNofxJ1x7TgphCzbh+htoLTf3GRWsXXetN7RDE4CnHfI/ERhombAr9X97P0aWMDbO9PoFlfb1fJ2RgsGt47FYnlp+dAEAXl2qtffIOBQzbWVn81KIj+gm9Uf8mO5bPXGDcbpG+Wb5nlSGmnWz8vqOvlNQb2fdL+k6OSYkH0DUEsDBBQAAAAIAAAAIVzgK8PjNwIAAPcDAAAWAAAAc3ViX2FnZW50cy9yZWFkZXIueWFtbG1TPXPbMAzd9SvQZGhyZyl24jipOnRq13bs5mMkSOKZIhkCjKzr9b8XpOwk7XWSCJIP7wO8hK8/f5Tr+1u4Xd/uys2mXD/UEFC1wM4ZAmfNDE8zaEscYsPa2QpCtPvGjaOyLWgCBV1ABBrQGLiiCXuUPcAjNjFf8HO9fbheATm4sA6w1bzAX8DgTEupAQ8IPrjRMyRY6/hcTSfBaOLPeZnIadsTNAM2h9cSho9SOpGyakSqikuh3ZjYorC1jJYJWuxUNFy/u5ckdDoQZ4JaDk3atm5K9dxf0QEEj1SPgFpKASY1rzJPdUY8Xzog+v/c091SS0YJLr4ISMBSnISrHMF2Dd82GzCOWDhAtImZdRavkw4X2UfeH3AGb5Rd6HfaKgOMR4YrHlzsBwavggjAY5bdrsCYcS/9LacQHu8/lY+7u+skXNxbaE5BMwbKatJaeW+09OYhJMhc+5WafvktvUb5Y1zBNOhmSEoCPkUt8jvJDohlU2Ym65tl7zkiLYGKtRTDi37BHJNXyywVKar6lETRIjVB+7RTw3ePdqHYuFasGxTDGAWuGZQVSxNqQO+S4HRKE0VhtjhjTn94lEbQxBDEAzneC/SrVJ2GQmw6IVbFKI1MDXl+y22J26dSc/msuJy2arMrGy7ePYQaPpzmC6rqZhleulmkVGNb/Dt89XlWirc465xnkV9DXQCU2Yp9ErCs3p5aXvfI++RypEJSxSB+n/HTt9P936yWGt2QkuTk2VSzGk3xB1BLAQIUABQAAAAIAAAAIVyPXcNCvgEAADEDAAAKAAAAAAAAAAAAAACkAQAAAABhZ2VudC55YW1sUEsBAhQAFAAAAAgAAAAhXNj8oRjVAAAAGAEAABUAAAAAAAAAAAAAAKQB5gEAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUABQAAAAIAAAAIVzoDefWxAEAABkDAAAQAAAAAAAAAAAAAACkAe4CAABldmFsX2NvbmZpZy55YW1sUEsBAhQAFAAAAAgAAAAhXGrfVvMkBwAAbw8AABIAAAAAAAAAAAAAAKQB4AQAAHByb21wdHMvYXBwbGllci5tZFBLAQIUABQAAAAIAAAAIVzDViR6awIAAIAEAAAUAAAAAAAAAAAAAACkATQMAABwcm9tcHRzL2RyYWZ0ZXJfMS5tZFBLAQIUABQAAAAIAAAAIVzHoQ60pAIAAPAEAAAUAAAAAAAAAAAAAACkAdEOAABwcm9tcHRzL2RyYWZ0ZXJfMi5tZFBLAQIUABQAAAAIAAAAIVwcMWocggIAAKQEAAAUAAAAAAAAAAAAAACkAacRAABwcm9tcHRzL2RyYWZ0ZXJfMy5tZFBLAQIUABQAAAAIAAAAIVyZg86QoAIAAOYEAAAUAAAAAAAAAAAAAACkAVsUAABwcm9tcHRzL2RyYWZ0ZXJfNC5tZFBLAQIUABQAAAAIAAAAIVwAY53spwIAAAMFAAAUAAAAAAAAAAAAAACkAS0XAABwcm9tcHRzL2RyYWZ0ZXJfNS5tZFBLAQIUABQAAAAIAAAAIVx9jjqO+gQAADwKAAARAAAAAAAAAAAAAACkAQYaAABwcm9tcHRzL3JlYWRlci5tZFBLAQIUABQAAAAIAAAAIVx5tw9abwIAAHMEAAAXAAAAAAAAAAAAAACkAS8fAABzdWJfYWdlbnRzL2FwcGxpZXIueWFtbFBLAQIUABQAAAAIAAAAIVyjEUSxowEAAE4DAAAVAAAAAAAAAAAAAACkAdMhAABzdWJfYWdlbnRzL2RyYWZ0LnlhbWxQSwECFAAUAAAACAAAACFcCb8oGcYBAAAlAwAAGQAAAAAAAAAAAAAApAGpIwAAc3ViX2FnZW50cy9kcmFmdGVyXzEueWFtbFBLAQIUABQAAAAIAAAAIVzmziXBHQEAAOEBAAAZAAAAAAAAAAAAAACkAaYlAABzdWJfYWdlbnRzL2RyYWZ0ZXJfMi55YW1sUEsBAhQAFAAAAAgAAAAhXBcAT9MdAQAA4QEAABkAAAAAAAAAAAAAAKQB+iYAAHN1Yl9hZ2VudHMvZHJhZnRlcl8zLnlhbWxQSwECFAAUAAAACAAAACFcwTtzQiUBAAADAgAAGQAAAAAAAAAAAAAApAFOKAAAc3ViX2FnZW50cy9kcmFmdGVyXzQueWFtbFBLAQIUABQAAAAIAAAAIVx8rCm2JwEAAPkBAAAZAAAAAAAAAAAAAACkAaopAABzdWJfYWdlbnRzL2RyYWZ0ZXJfNS55YW1sUEsBAhQAFAAAAAgAAAAhXOArw+M3AgAA9wMAABYAAAAAAAAAAAAAAKQBCCsAAHN1Yl9hZ2VudHMvcmVhZGVyLnlhbWxQSwUGAAAAABIAEgCxBAAAcy0AAAAA"               # filled in by tools/build_notebook.py --embed
# Model: taken from the embedded zip's agent.yaml. Set MODEL_NAME only as a cross-check (the run asserts the
# zip declares exactly this name), e.g. --set MODEL_NAME='"gemma-4-e2b-it-qat-w4a16-ct"'. The kernel-metadata
# model_sources must attach google/gemma-4/other/<name>/2 for the same name.
MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"   # overridden by --set

# Compaction settings: Q7 ANSWERED [V] from HARNESS_README.md section 7.2 (interval 15, overlap 2,
# token_threshold 14,336, event_retention_size 5). The 'interval 5' reading confused retention with interval.
COMPACTION_INTERVAL = 15   # [V] HARNESS_README §7.2: interval 15, overlap 2, threshold 14,336, retention 5
COMPACTION_TOKEN_THRESHOLD = 14336

In [ ]:
# ---------------------------------------------------------------- environment + wheelhouse (hosts' recipe)
# Adapted from the organizers' Getting-Started notebook as mirrored by community repos.  # VERIFY
import glob, importlib, json, os, shutil, subprocess, sys, time, zipfile, io, base64
from pathlib import Path

os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1', 'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
WHEELHOUSE_DIR = next(Path(p) for p in ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
                                        '/kaggle/input/gemma-4-developer-agent-wheelhouse'] if Path(p).exists())
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
# The wheelhouse ships cp312 binary wheels (orchestrator env = Python 3.12), but the grading sandbox and
# Kaggle's CPU image are Python 3.13 (docker/Dockerfile.sandbox is `python:3.13-slim`). A single bulk
# `pip install ... check=True` aborts the WHOLE install when one wheel is incompatible, which is what killed
# the first EXP-000 push (apache_tvm_ffi cp312 on py3.13). Filter by interpreter tag, then assert that the
# packages we actually need for this run arrived.
# Import names, not distribution names: the `google_adk` wheel installs the `google.adk`
# namespace package, so importing 'google_adk' always fails. [V] google_adk-1.36.1.dist-info
REQUIRED_PKGS = ('swegemma', 'adk_submission', 'adk_eval_core', 'google.adk')

def _wheel_is_compatible(path: Path) -> bool:
    """True when the wheel's tags intersect this interpreter's supported tags."""
    try:
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
    except Exception:  # noqa: BLE001 - fall back to accepting everything
        return True
    try:
        _, _, _, tags = parse_wheel_filename(path.name)
    except Exception:  # noqa: BLE001 - odd filename: let pip decide
        return True
    supported = set(sys_tags())
    return any(t in supported for t in tags)

all_wheels = sorted(tmp_whl.glob('*.whl'))
usable = [w for w in all_wheels if _wheel_is_compatible(w)]
skipped = [w.name for w in all_wheels if w not in usable]
T_INSTALL = time.time()
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                       *(str(w) for w in usable)], capture_output=True, text=True)
if proc.returncode != 0:
    print('bulk install failed; retrying wheel-by-wheel to isolate the bad ones')
    failed = []
    for w in usable:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            failed.append(w.name)
    skipped += failed
importlib.invalidate_caches()
print(f'wheelhouse: installed {len(all_wheels) - len(skipped)}/{len(all_wheels)} wheels in {time.time() - T_INSTALL:.0f}s '
      f'(python {sys.version_info.major}.{sys.version_info.minor})')
if skipped:
    print(f'  incompatible/failed ({len(skipped)}):', ', '.join(sorted(skipped)[:12]))
missing = []
for _pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(_pkg)
    except Exception as _exc:  # noqa: BLE001
        missing.append(f'{_pkg} ({_exc.__class__.__name__}: {_exc})')
assert not missing, 'required packages unavailable: ' + '; '.join(missing)
print('  required packages present:', ', '.join(REQUIRED_PKGS))

DATA_DIR = next(Path(p) for p in ['/kaggle/input/competitions/gemma-4-developer-agent',
                                  '/kaggle/input/gemma-4-developer-agent'] if Path(p).exists())
RUN_DIR = Path('/kaggle/working/runs') / EXP_ID
for sub in ('traces', 'patches', 'logs', 'phase0'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, ' run dir:', RUN_DIR)

In [ ]:
# ---------------------------------------------------------------- Phase 0 checks (plan §3.4)
# Answers Q1/Q2/Q3/Q7/Q9 by reading the installed harness source; saves HARNESS_README.md and
# sample_submission/ to the run dir so they can be downloaded and read offline.
if RUN_PHASE0_CHECKS:
    import importlib.metadata as md, re
    report = {'python': sys.version, 'packages': {}}
    for pkg in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm'):
        try:
            report['packages'][pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            report['packages'][pkg] = None
    for name in ('HARNESS_README.md',):
        src = DATA_DIR / name
        if src.exists():
            shutil.copy(src, RUN_DIR / 'phase0' / name)
    ss = DATA_DIR / 'sample_submission'
    if ss.exists():
        shutil.copytree(ss, RUN_DIR / 'phase0' / 'sample_submission', dirs_exist_ok=True)
        print('--- sample_submission files (Q2) ---')
        for p in sorted(ss.rglob('*')):
            if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.json'):
                print(f'===== {p.relative_to(ss)}'); print(p.read_text()[:4000])

    def grep_pkg(pkg: str, patterns: list[str]) -> dict:
        hits = {}
        try:
            root = Path(importlib.import_module(pkg).__file__).parent
        except Exception as exc:  # noqa: BLE001
            return {'error': str(exc)}
        for pat in patterns:
            rx = re.compile(pat)
            for f in root.rglob('*.py'):
                for i, line in enumerate(f.read_text(errors='ignore').splitlines(), 1):
                    if rx.search(line):
                        hits.setdefault(pat, []).append(f'{f.relative_to(root)}:{i}: {line.strip()[:160]}')
        return hits

    report['Q1_tool_registry'] = grep_pkg('swegemma', [r'register\(', r'def (run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph)\b'])
    report['Q1_skill_tools'] = grep_pkg('adk_submission', [r'run_skill_script|load_skill_resource|SkillToolset'])
    report['Q2_eval_config_keys'] = grep_pkg('swegemma', [r'max_time_minutes|max_tool_calls|max_turns|timeout_seconds', r'load_submission_eval_config'])
    report['Q2_agent_tool_schema'] = grep_pkg('adk_submission', [r'agent_tool|config_path|skip_summarization'])
    report['Q3_thinking_forwarding'] = grep_pkg('adk_submission', [r'thinking_budget|thinking_token_budget|enable_thinking|include_thoughts'])
    report['Q7_compaction'] = grep_pkg('swegemma', [r'compaction_interval|token_threshold|EventsCompactionConfig'])
    report['Q9_extensions_limits'] = grep_pkg('swegemma', [r'ALLOWED_SUBMISSION_EXTENSIONS|MAX_SUBMISSION_SIZE_BYTES|allowed_file_extensions|max_total_size_bytes'])
    report['Q2_grading_timeout'] = grep_pkg('swegemma', [r'timeout', ]) if False else grep_pkg('swegemma', [r'verification.*timeout|timeout.*verif|command_timeout_seconds'])
    (RUN_DIR / 'phase0' / 'phase0_report.json').write_text(json.dumps(report, indent=1))
    print(json.dumps({k: (v if k == 'packages' else {kk: len(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
                      for k, v in report.items()}, indent=1))
    print('full grep hits saved to', RUN_DIR / 'phase0' / 'phase0_report.json')

In [ ]:
# ---------------------------------------------------------------- tasks + candidates
from swegemma.models import load_tasks  # [V] swegemma/models/__init__.py exports load_tasks(path)->list[Task]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
from collections import Counter
print('repo counts:', Counter(t.repo for t in all_tasks))
if TASK_IDS:
    want = set(TASK_IDS)
    candidates = [t for t in all_tasks if t.instance_id in want]
else:
    pool = [t for t in all_tasks if (not REPO_FILTER or t.repo == REPO_FILTER) and t.instance_id not in set(EXCLUDE_IDS)]
    candidates = sorted(pool, key=lambda t: t.instance_id)[:N_TASKS]
print(f'{len(all_tasks)} tasks; candidates ({len(candidates)}):', [t.instance_id for t in candidates])

In [ ]:
# ---------------------------------------------------------------- unpack our submission
AGENT_DIR = Path('/kaggle/working/agent')
shutil.rmtree(AGENT_DIR, ignore_errors=True); AGENT_DIR.mkdir(parents=True)
if SUBMISSION_B64:
    zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)
elif Path(SUBMISSION_ZIP).exists():
    zipfile.ZipFile(SUBMISSION_ZIP).extractall(AGENT_DIR)
elif not RUN_AGENT and (DATA_DIR / 'sample_submission').is_dir():
    # A gold-control run never calls a model, so it needs no submission of ours. Fall back to the
    # official sample_submission: it is a real, schema-valid submission dir, so the model/adapter/limits
    # validation below still runs for real instead of being skipped.
    shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR, dirs_exist_ok=True)
    print('no submission supplied; using official sample_submission (RUN_AGENT=False)')
else:
    raise SystemExit('no submission: set SUBMISSION_ZIP to your uploaded dataset file or embed with tools/build_notebook.py')
print('agent files:', sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))
import yaml
EVAL = (yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) or {}).get('evaluation', {}) \
    if (AGENT_DIR / 'eval_config.yaml').exists() else {}
if not RUN_AGENT:
    # The official sample eval_config is deliberately tiny (1 min / 10 tool calls) to demo the schema.
    # Gold control runs no agent loop, so those caps only risk failing a sound task for budget reasons
    # and corrupting the Q5 answer. Give the grader room; the agent budget is irrelevant here.
    EVAL = {**EVAL, 'timeout_seconds': 300, 'max_time_minutes': 60.0, 'max_tool_calls': 100}
print('eval_config:', EVAL)

In [ ]:
# ---------------------------------------------------------------- vLLM server (hosts' recipe)  # VERIFY
# CPU-safe imports only: a gold-control run (RUN_AGENT=False) never loads vLLM/litellm/torch, and on
# Kaggle's Python 3.13 CPU image those cp312 wheels are not installed at all.
from adk_submission import ModelRegistry, discover_adapters, discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits

# D-004 2026-11-05: TARGET was hard-coded to 31B; it now comes from the zip, so a small-variant candidate
# runs without editing this file. Mirror of tools/submission_rules.py ALLOWED_MODEL_NAMES (the notebook runs
# on Kaggle and cannot import tools/). 12B is excluded there and here: it needs vLLM --hf-overrides
# (HARNESS_README §3.1 MODEL_VLLM_EXTRA_ARGS), which this cell does not pass.
ALLOWED_MODELS = ('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct')
# [V] adk_submission.discovery.discover_declared_models(submission_dir) -> set[str]; walks the root
# agent, sub_agents and agent_tools. There is no `validate_single_declared_model` anywhere in the
# wheelhouse -- that was an invented name. Exactly one model may be declared, so assert the set size.
declared = discover_declared_models(str(AGENT_DIR))
assert len(declared) == 1, f'submission must declare exactly one model, got {sorted(declared)}'
TARGET = next(iter(declared)).strip()
for _prefix in ('openai/', 'google/', 'hosted_vllm/', 'custom/'):   # [V] HARNESS_README §3.2 normalize_model_name
    if TARGET.startswith(_prefix):
        TARGET = TARGET[len(_prefix):]
        break
assert TARGET in ALLOWED_MODELS, f'declared model {TARGET!r} not in {ALLOWED_MODELS}'
assert not MODEL_NAME or MODEL_NAME == TARGET, f'MODEL_NAME={MODEL_NAME!r} but the zip declares {TARGET!r}'
MODEL_DIR = f'/kaggle/input/models/google/gemma-4/other/{TARGET}/2'   # [V] path layout EXP-006 resolved 'exact'
print('model:', TARGET, '->', MODEL_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
# EXP-000 gold control needs no model: it applies each task's reference patch and runs test_patch.
# Gate the server so a gold-control-only run costs ZERO GPU quota (plan §5.7) and can use a CPU notebook.
NEED_SERVER = RUN_AGENT or MEASURE_DECODE_SPEED
server = None
SERVER_START_S = None
target_id = None
models = ModelRegistry()

if NEED_SERVER:
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    from google.adk.models.lite_llm import LiteLlm
    try:
        from adk_submission.resolvers.generation import install_litellm_reasoning_patch
        install_litellm_reasoning_patch()
    except Exception as exc:  # noqa: BLE001
        print('reasoning patch not installed:', exc)
    litellm.drop_params = True
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    assert gpu_count >= 1, 'no GPU: select the L4 x4 accelerator'
    assert Path(MODEL_DIR, 'config.json').exists(), (
        f'{MODEL_DIR} missing: kernel-metadata.json model_sources must list google/gemma-4/other/{TARGET}/2')
    tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    t0 = time.time()
    server = VllmServer(VllmConfig(
        model=MODEL_DIR, port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        default_chat_template_kwargs={'enable_thinking': True}, max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
        tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
    server.start()
    SERVER_START_S = round(time.time() - t0, 1)
    print(f'vLLM up in {SERVER_START_S}s (tp={tp}, gpus={gpu_count}) at {server.base_url}')
    target_id = getattr(server, 'effective_model_path', MODEL_DIR)
    for alias in {next(iter(declared)), TARGET}:
        models.register(alias, LiteLlm(model=f'openai/{target_id}', api_base=server.base_url, api_key='EMPTY'))
else:
    print('vLLM server SKIPPED (RUN_AGENT and MEASURE_DECODE_SPEED are both False) - no GPU required.')

In [ ]:
# ---------------------------------------------------------------- Q10: decode speed probe
DECODE = None
if MEASURE_DECODE_SPEED:
    import urllib.request
    body = json.dumps({'model': target_id, 'messages': [{'role': 'user', 'content': 'Count from 1 to 400 separated by spaces.'}],
                       'max_tokens': 400, 'temperature': 0.0, 'chat_template_kwargs': {'enable_thinking': False}}).encode()
    req = urllib.request.Request(server.base_url.rstrip('/') + '/chat/completions', data=body,
                                 headers={'Content-Type': 'application/json'}, method='POST')
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as resp:
        out = json.loads(resp.read())
    dt = time.time() - t0
    ct = out.get('usage', {}).get('completion_tokens', 0)
    DECODE = {'completion_tokens': ct, 'seconds': round(dt, 2), 'tok_s': round(ct / dt, 1) if dt else None}
    print('decode probe:', DECODE)

In [ ]:
# ---------------------------------------------------------------- evaluator helpers
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig  # [V] google/adk/apps/_configs.py:50
from swegemma.evaluate import Evaluator

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def make_config(results_dir: Path) -> EvalConfig:
    limits, gen = build_submission_limits()
    return EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
        timeout_seconds=int(EVAL.get('timeout_seconds', 300)),
        max_time_minutes=float(EVAL.get('max_time_minutes', 60.0)),
        max_tool_calls=int(EVAL.get('max_tool_calls', 100)),
        max_turns=int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=COMPACTION_INTERVAL, overlap_size=2,
                                                        token_threshold=COMPACTION_TOKEN_THRESHOLD, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False)

def record(r, task, t_start: float, t_end: float) -> dict:
    patch = r.agent_patch or ''
    dur = float(r.duration_seconds or 0)
    wall = t_end - t_start
    return {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
            'status': getattr(r, 'status', None), 'test_exit_code': r.test_exit_code,
            'duration_s': round(dur, 1), 'task_wall_s': round(wall, 1),
            'setup_s': round(max(0.0, wall - dur), 1),   # everything outside the agent loop (setup + grading)  VERIFY
            'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
            'total_tokens': getattr(r, 'total_tokens', None), 'error': getattr(r, 'error', None) or getattr(r, 'error_message', None),
            'patch_chars': len(patch), 'patch': patch, 'trace': str(getattr(r, 'trace_path', '') or getattr(r, 'trace_json_path', '') or ''),
            'test_tail': (r.test_output or '')[-1500:], 'max_time_minutes': EVAL.get('max_time_minutes'),
            'max_tool_calls': EVAL.get('max_tool_calls')}

def append_row(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as fh:
        fh.write(json.dumps(row) + '\n')

In [ ]:
# ---------------------------------------------------------------- EXP-000 gold control (Q5)
# `sound` must exist even when the gold control is skipped, or the agent cell below raises NameError.
sound = list(candidates)
if RUN_GOLD_CONTROL:
    from adk_eval_core.tracing import SessionTrace  # [V] tracing/__init__.py re-exports it; no-arg ctor

    class GoldEvaluator(Evaluator):
        """Phase 1 returns the reference patch; no model call, nothing is shown to any agent."""
        async def _run_agent_sandbox(self, task, *args, **kwargs):  # [V] evaluate.py:194 -> tuple[str, str|None, SessionTrace]
            return task.patch, None, SessionTrace()

    gold = GoldEvaluator(make_config(RUN_DIR / 'gold_control'))
    gold_rows = []
    for i, t in enumerate(candidates, 1):
        t0 = time.time()
        r = run_sync(lambda: gold.evaluate_task(task=t, task_index=i, total_tasks=len(candidates)))
        row = {'instance_id': t.instance_id, 'repo': t.repo, 'mode': 'gold', 'resolved': bool(r.resolved),
               'exit': r.test_exit_code, 'status': getattr(r, 'status', None), 'seconds': round(time.time() - t0, 1), 'error': (getattr(r, 'error', '') or '')[:300],
               'tail': (r.test_output or '')[-600:]}
        gold_rows.append(row)
        append_row(RUN_DIR / 'gold_control.jsonl', row)
        print('gold', t.instance_id, 'resolved=', row['resolved'], f"{row['seconds']}s")
    failures = [x['instance_id'] for x in gold_rows if not x['resolved']]
    (RUN_DIR / 'gold_failures.txt').write_text('# gold-control failures (exclude from cohorts)\n' + '\n'.join(failures) + '\n')
    sound = [t for t, x in zip(candidates, gold_rows) if x['resolved']]
    print(f'gold control: {len(sound)}/{len(candidates)} sound; failures: {failures}')

In [ ]:
# ---------------------------------------------------------------- baseline agent eval
summary = {'exp_id': EXP_ID, 'server_start_s': SERVER_START_S, 'decode_probe': DECODE, 'eval_config': EVAL,
           'compaction': {'interval': COMPACTION_INTERVAL, 'token_threshold': COMPACTION_TOKEN_THRESHOLD},
           'tasks': [], 'resolved': 0}
if RUN_AGENT:
    ev = Evaluator(make_config(RUN_DIR / 'official'))
    for i, t in enumerate(sound, 1):
        t0 = time.time()
        r = run_sync(lambda: ev.evaluate_task(task=t, task_index=i, total_tasks=len(sound)))
        row = record(r, t, t0, time.time())
        append_row(RUN_DIR / 'task_results.jsonl', row)
        (RUN_DIR / 'patches' / f'{t.instance_id}.patch').write_text(row['patch'])
        for cand in (Path(row['trace']) if row['trace'] else None, RUN_DIR / 'official' / 'traces' / f'trace_{t.instance_id}.json'):
            if cand and cand.exists():
                shutil.copy(cand, RUN_DIR / 'traces' / f'trace_{t.instance_id}.json'); break
        summary['tasks'].append(t.instance_id); summary['resolved'] += int(row['resolved'])
        print(f"[{i}/{len(sound)}] {t.instance_id} resolved={row['resolved']} calls={row['tool_calls']} "
              f"agent={row['duration_s']}s setup={row['setup_s']}s err={(row['error'] or '')[:100]}")
    n = len(summary['tasks'])
    print(f"RESOLVED {summary['resolved']}/{n}" if n else 'no tasks run')
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))

In [ ]:
# ---------------------------------------------------------------- bundle for download + stop server
# Official logs may contain repository snippets: keep the bundle private, never commit it (plan §1.3).
for extra in (RUN_DIR / 'official' / 'logs',):
    if extra.exists():
        shutil.copytree(extra, RUN_DIR / 'logs', dirs_exist_ok=True)
shutil.make_archive(str(Path('/kaggle/working/runs') / EXP_ID), 'zip', RUN_DIR)
print('bundle:', Path('/kaggle/working/runs') / f'{EXP_ID}.zip')
try:
    server.stop()
except Exception:  # noqa: BLE001
    pass